In [176]:
import pandas as pd
import nltk
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

## Preprocessing

#### Merge all text columns into one

In [177]:
def merge_cols(row, columns):
    merged = ''
    is_na = row.isna()
    for col in columns:
        if not is_na[col]:
            merged = merged + row[col] + ' '
    merged = merged[:-1]
    return merged

In [178]:
def merge_all(df=None, in_file=None, out_file=None):
    if df is None:
        if in_file is None:
            return
        df = pd.read_csv(in_file, encoding = 'ISO-8859-1')
    text_cols = df.columns[1:]
    out_df = pd.DataFrame(df.iloc[:, 0].values, columns=['class'])
    out_df['body'] = df.apply(lambda row: merge_cols(row, text_cols), axis='columns')
    if out_file is not None:
        out_df.to_csv(out_file, index=False)
    return out_df

In [179]:
_ = merge_all(in_file='data/spam.csv', out_file='data/mergedSpam.csv')

### Get Frequencies

In [180]:
def tokenize(text):
    # Remove single periods, commas, apostrophes, quotes, and parentheses
    text = re.sub(r'(?<!\.)\.(?!\.)|,|\'|\"|\(|\)|‘|’|“|”', '', text)
    
    # Use nltk to split into tokens
    tokens = nltk.word_tokenize(text)
    
    # I'm leaving other punctuation since it can be an indicator of being spam, such as many exclamation marks, etc.
    # nltk tokenize should mostly split those punctuations into their own tokens
    return tokens

In [181]:
def vectorize(df=None, in_file=None, out_file=None, vectorizer_func=CountVectorizer):
    """
    Expects a dataframe that has been merged. This should only be called on training set
    Use vectorizer_transform_df after to vectorize testing set. (This doesnt really matter for wordcount but does for tfidf)
    :param df: 
    :param in_file: 
    :param out_file: 
    :param vectorizer_func: 
    :return: 
    """
    if df is None:
        if in_file is None:
            return
        df = pd.read_csv(in_file, encoding = 'ISO-8859-1')
    vectorizer = vectorizer_func(tokenizer=tokenize, token_pattern=None, stop_words='english')
    vectors = vectorizer.fit_transform(df['body'])
    out_df = pd.DataFrame.sparse.from_spmatrix(vectors, columns=vectorizer.get_feature_names_out())
    # Since class could be a word, I am using .class.
    df.reset_index(drop=True, inplace=True)
    out_df['.class.'] = df['class']
    if out_file is not None:
        out_df.to_csv(out_file, index=False)
    return out_df, vectorizer

In [182]:
def vectorizer_transform_df(vectorizer, df=None, in_file=None, out_file=None):
    if df is None:
        if in_file is None:
            return
        df = pd.read_csv(in_file, encoding = 'ISO-8859-1')
    vectors = vectorizer.transform(df['body'])
    out_df = pd.DataFrame.sparse.from_spmatrix(vectors, columns=vectorizer.get_feature_names_out())
    df.reset_index(drop=True, inplace=True)
    out_df['.class.'] = df['class']
    if out_file is not None:
        out_df.to_csv(out_file, index=False)
    return out_df

# Multinomial NB

In [183]:
def train(df=None, in_file=None, classifier=MultinomialNB, **kwargs):
    if df is None:
        if in_file is None:
            return
        df = pd.read_csv(in_file)
    clf = classifier(**kwargs)
    clf.fit(df.drop(columns=['.class.']), df['.class.'])
    return clf

In [184]:
def test(df=None, in_file=None, vectorizer_func=CountVectorizer, frac=0.2, classifier=MultinomialNB, **kwargs):
    """
    Expects a dataframe that has been merged but not vectorized
    :param df: 
    :param in_file: 
    :param vectorizer_fun: 
    :param frac: 
    :param classifier: 
    :param kwargs: 
    :return: 
    """
    if df is None:
        if in_file is None:
            return
        df = pd.read_csv(in_file)
    train_df = df.sample(frac=frac)
    #display(train_df)
    test_df = df.drop(train_df.index)
    train_df, vectorizer = vectorize(train_df, vectorizer_func=vectorizer_func)
    #display(train_df)
    test_df = vectorizer_transform_df(vectorizer, test_df)
    clf = train(df=train_df, classifier=classifier, **kwargs)
    training_accuracy = clf.score(train_df.drop(columns=['.class.']), train_df['.class.'])
    testing_accuracy = clf.score(test_df.drop(columns=['.class.']), test_df['.class.'])
    print(f'Overall Accuracy')
    print(f'Training Accuracy: {training_accuracy:.2f}')
    print(f'Testing Accuracy: {testing_accuracy:.2f}')
    train_proportions = train_df['.class.'].value_counts(normalize=True)
    test_proportions = test_df['.class.'].value_counts(normalize=True)
    class_labels = ['ham', 'spam']
    for label in class_labels:
        label_train_df = train_df.loc[train_df['.class.'] == label]
        label_test_df = test_df.loc[test_df['.class.'] == label]
        label_train_accuracy = clf.score(label_train_df.drop(columns=['.class.']), label_train_df['.class.'])
        label_test_accuracy = clf.score(label_test_df.drop(columns=['.class.']), label_test_df['.class.'])
        print(f'{label} Stats:')
        print(f'Proportion of Training Set: {train_proportions[label]:.2f}')
        print(f'Training Accuracy: {label_train_accuracy:.2f}')
        print(f'Proportion of Test Set: {test_proportions[label]:.2f}')
        print(f'Testing Accuracy: {label_test_accuracy:.2f}')

In [185]:
print('Word Counts')
_ = test(in_file='data/mergedSpam.csv', vectorizer_func=CountVectorizer)
print('---------')
print('TFIDF')
_ = test(in_file='data/mergedSpam.csv', vectorizer_func=TfidfVectorizer)

Word Counts
Overall Accuracy
Training Accuracy: 1.00
Testing Accuracy: 0.98
ham Stats:
Proportion of Training Set: 0.85
Training Accuracy: 1.00
Proportion of Test Set: 0.87
Testing Accuracy: 0.99
spam Stats:
Proportion of Training Set: 0.15
Training Accuracy: 0.98
Proportion of Test Set: 0.13
Testing Accuracy: 0.86
---------
TFIDF
Overall Accuracy
Training Accuracy: 0.96
Testing Accuracy: 0.92
ham Stats:
Proportion of Training Set: 0.87
Training Accuracy: 1.00
Proportion of Test Set: 0.86
Testing Accuracy: 1.00
spam Stats:
Proportion of Training Set: 0.13
Training Accuracy: 0.70
Proportion of Test Set: 0.14
Testing Accuracy: 0.38


## KNN

In [188]:
print('Word Counts')
_ = test(in_file='data/mergedSpam.csv', vectorizer_func=CountVectorizer, classifier=KNeighborsClassifier, n_neighbors=5)
print('---------')
print('TFIDF')
_ = test(in_file='data/mergedSpam.csv', vectorizer_func=TfidfVectorizer, classifier=KNeighborsClassifier, n_neighbors=5)

Word Counts
Overall Accuracy
Training Accuracy: 0.89
Testing Accuracy: 0.88
ham Stats:
Proportion of Training Set: 0.87
Training Accuracy: 1.00
Proportion of Test Set: 0.87
Testing Accuracy: 1.00
spam Stats:
Proportion of Training Set: 0.13
Training Accuracy: 0.16
Proportion of Test Set: 0.13
Testing Accuracy: 0.08
---------
TFIDF
Overall Accuracy
Training Accuracy: 0.96
Testing Accuracy: 0.93
ham Stats:
Proportion of Training Set: 0.89
Training Accuracy: 1.00
Proportion of Test Set: 0.86
Testing Accuracy: 1.00
spam Stats:
Proportion of Training Set: 0.11
Training Accuracy: 0.61
Proportion of Test Set: 0.14
Testing Accuracy: 0.52
